# 1D Poisson-Nernst-Planck (PNP) System: PINN vs. Data-Driven Neural Network

**Author:** Jack  

## 1. Project Overview
This notebook evaluates two distinct neural network approaches against a classical numerical benchmark for solving the 1D Poisson-Nernst-Planck (PNP) equations:

1. **PINN vs. Benchmark:** We assess a Physics-Informed Neural Network (PINN) that learns the system dynamics purely by minimizing the PDE residuals and boundary/initial conditions, comparing its accuracy against an explicit Finite Difference Method (FDM) baseline.
2. **Regular NN vs. Benchmark:** We evaluate a standard, purely data-driven Neural Network trained explicitly on auxiliary data (the FDM ground truth) using supervised learning, against the benchmark.


The goal is to contrast how well each architecture captures the drift-diffusion of ions under an electric field, particularly near the boundary layers.

## 2. Governing Equations
The 1D PNP system consists of the Nernst-Planck equations for mass transport and the Poisson equation for electrostatics.

For cations ($c_p$) and anions ($c_n$), the fluxes are defined as:
$$J_p = -D_p \frac{\partial c_p}{\partial x} - D_p z_p c_p \frac{\partial \phi}{\partial x}$$
$$J_n = -D_n \frac{\partial c_n}{\partial x} - D_n z_n c_n \frac{\partial \phi}{\partial x}$$

The transient Nernst-Planck equations ensure conservation of mass:
$$\frac{\partial c_p}{\partial t} + \frac{\partial J_p}{\partial x} = 0$$
$$\frac{\partial c_n}{\partial t} + \frac{\partial J_n}{\partial x} = 0$$

The electrostatic potential ($\phi$) is governed by the Poisson equation, coupled to the charge density:
$$-\epsilon \frac{\partial^2 \phi}{\partial x^2} = z_p c_p + z_n c_n$$

**Domain & Boundary Conditions:**
* Spatial Domain: $x \in [0, 1]$
* Time Domain: $t \in [0, 0.2]$
* Blocking Electrodes (Zero Flux): $J_p = J_n = 0$ at $x=0, 1$
* Dirichlet Potential: $\phi(0, t) = -0.5$ and $\phi(1, t) = 0.5$

## 3. PINN Architecture & Training Setup

### Network Architecture
The surrogate model is a deep Multi-Layer Perceptron (MLP) mapping spatiotemporal inputs to the system's state variables.
* **Inputs:** Spatial coordinate $x$, Time $t$
* **Outputs:** Cation concentration ($c_p$), Anion concentration ($c_n$), Electrostatic potential ($\phi$)
* **Hidden Layers:** 8 layers
* **Neurons per Layer:** 256
* **Activation Function:** Hyperbolic Tangent (Tanh)

### Physics Constraints and Loss Weights
The PINN is trained by minimizing a composite loss function. Each physical constraint is enforced by sampling collocation points and applying specific loss weighting ($\lambda$) to guide the optimizer.

**1. Initial Condition (IC)**
* **Domain:** $t = 0$
* **Equations:** $c_p = 1.0$, $c_n = 1.0$
* **Weights:** $\lambda_{c_p} = 100.0$, $\lambda_{c_n} = 100.0$

**2. Global Interior (PDEs)**
* **Domain:** $x \in [0.0, 1.0]$, $t \in [0.0, 0.2]$
* **Equations:** Nernst-Planck ($c_p$, $c_n$) and Poisson residuals equal to $0$
* **Weights:** $\lambda = 1.0$ (for all three PDEs)

**3. Boundary Layer Interiors (High-Density PDEs)**

To capture steep concentration gradients near the electrodes, additional collocation points are packed into the 5% margins of the spatial domain.
* **Domains:** Left ($x \in [0.0, 0.05]$) and Right ($x \in [0.95, 1.0]$)
* **Equations:** Nernst-Planck ($c_p$, $c_n$) and Poisson residuals equal to $0$
* **Weights:** $\lambda = 10.0$ (for all three PDEs)

**4. Boundary Conditions: Zero Flux (Blocking Electrodes)**
* **Domain:** $x = 0.0$ and $x = 1.0$
* **Equations:** $J_p = 0.0$, $J_n = 0.0$
* **Weights:** $\lambda_{J_p} = 50.0$, $\lambda_{J_n} = 50.0$

**5. Boundary Conditions: Applied Potential (Dirichlet)**
* **Left Electrode ($x = 0.0$):** $\phi = -0.5$ ($\lambda_{\phi} = 50.0$)
* **Right Electrode ($x = 1.0$):** $\phi = 0.5$ ($\lambda_{\phi} = 50.0$)

## 4. Supervised NN Architecture & Training Setup

### Network Architecture
To ensure a fair **1:1 comparison**, the standard Neural Network (NN) uses an **identical structure to the PINN**.

* **Inputs:** Spatial coordinate $x$, Time $t$
* **Outputs:** Cation concentration ($c_p$), Anion concentration ($c_n$), Electrostatic potential ($\phi$)
* **Hidden Layers:** 8 layers
* **Neurons per Layer:** 256
* **Activation Function:** Hyperbolic Tangent (Tanh)

### Supervised Training Setup
The standard NN is a **purely data-driven model**. Instead of enforcing physics constraints, it learns the mapping from inputs to outputs using **benchmark numerical solutions**.

**1. Training Data (Auxiliary Data)**
* **Source:** Ground truth results generated by the **explicit Finite Difference Method (FDM)** solver.
* **Method:** The model is trained directly on these pre-computed solutions to learn the mapping  
  $(x, t) \rightarrow (c_p, c_n, \phi)$.

**2. Loss Function**
* **Objective:** Minimize the **Mean Squared Error (MSE)** between the neural network predictions and the **FDM benchmark values**.

**3. Model Capability**
* **Finding:** The supervised NN achieved **significantly lower error than the PINN**.
* **Interpretation:** This confirms that the **network architecture (8 layers, 256 neurons)** has sufficient capacity to represent the **PNP solution manifold**. Therefore, the performance gap is not due to architectural limitations, but rather the **difficulty of enforcing physics constraints during PINN training**.

## 4. FDM Benchmark Setup (Ground Truth Generation)

To establish a rigorous ground truth for evaluating the neural networks, the 1D PNP system was solved numerically using a custom explicit Finite Difference Method (FDM) solver. The solver is designed to strictly conserve mass and enforce blocking boundary conditions.

### Spatiotemporal Discretization
* **Spatial Grid:** The 1D domain is uniformly discretized into 101 nodes ($N=101$), resulting in a grid spacing of $dx = 0.01$.
* **Time Stepping:** To maintain numerical stability and satisfy the Courant–Friedrichs–Lewy (CFL) condition for an explicit diffusion solver, a microscopic time step of $dt = 10^{-5}$ seconds is used. The simulation runs for a total of 0.2 seconds (20,000 steps).

### Numerical Scheme
The simulation employs a staggered-grid approach, decoupling the electrostatics from the mass transport at each time step:

**1. Electrostatics (Implicit Poisson Solve)**
The Poisson equation is discretized using a central difference scheme:
$$\epsilon \frac{\phi_{i-1} - 2\phi_i + \phi_{i+1}}{dx^2} = -(c_{p,i} - c_{n,i})$$
Because the Laplacian operator is linear and time-invariant, the coefficient matrix $A$ (incorporating the Dirichlet boundary conditions $\phi_0 = -0.5$ and $\phi_{N-1} = 0.5$) is constructed and inverted exactly *once* before the simulation begins. At each time step, the new potential field is efficiently computed via a simple matrix-vector multiplication: $\phi = A^{-1} \cdot \text{rhs}$.

**2. Flux Calculation (Staggered Edges)**

To prevent numerical oscillations, the fluxes are calculated at the *edges* halfway between the spatial nodes ($i$ and $i+1$).
* The concentration gradients and potential gradients are calculated as simple forward differences: $\frac{\partial c}{\partial x} \approx \frac{c_{i+1} - c_i}{dx}$.
* The local concentration driving the drift flux is averaged between the adjacent nodes: $c_{\text{edge}} = \frac{c_{i+1} + c_i}{2}$.
* Total edge flux is then explicitly calculated:
$$J_{\text{edge}} = -D \frac{\partial c}{\partial x} - z c_{\text{edge}} \frac{\partial \phi}{\partial x}$$

**3. Mass Transport (Explicit Euler Update)**
Conservation of mass is enforced by computing the net flux into each node. Blocking electrodes are strictly enforced by padding the outer edges of the flux arrays with exact zeros ($J = 0$).
The concentrations are then stepped forward in time using an explicit Euler update:
$$c_{i}^{t+dt} = c_{i}^{t} - \frac{J_{i}^{\text{edge}} - J_{i-1}^{\text{edge}}}{dx} dt$$

### Data Extraction
To generate the training and validation dataset (`ground_truth.csv`), the state variables ($c_p$, $c_n$, $\phi$) at all 101 spatial nodes are recorded every 0.01 seconds, yielding a dataset of roughly 2,100 high-fidelity spatiotemporal data points.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

# --- 1. Setup Paths ---
PINN_PATH = "./outputs/PNP/outputs_PNP/inferencers/inf_end.npz"
NN_PATH = "./outputs/PNP_supervised/outputs_PNP/inferencers/inf_end.npz"
FIPY_PATH = "./data/ground_truth.csv"

print("Loading data...")

# --- 2. Load PINN Data ---
raw_data_pinn = np.load(PINN_PATH, allow_pickle=True)
pinn_dict = raw_data_pinn['arr_0'].item()

# --- 3. Load Supervised NN Data ---
raw_data_nn = np.load(NN_PATH, allow_pickle=True)
nn_dict = raw_data_nn['arr_0'].item()

# --- 4. Load FDM Benchmark Data ---
fipy_df = pd.read_csv(FIPY_PATH)
fipy_pts = fipy_df[['x', 't']].values

print("Data loaded successfully!")

In [ ]:
# --- 1. Prepare Grid (100x100 spatial/temporal shape based on inferencer) ---
X_grid = pinn_dict['x'].reshape(400, 100)
T_grid = pinn_dict['t'].reshape(400, 100)

# --- 2. Initialize Plot (Expanded to 5 columns) ---
fig, axes = plt.subplots(3, 5, figsize=(24, 12), sharex=True, sharey=True)
variables = ['cp', 'cn', 'phi']
titles = ['Cations ($c_p$)', 'Anions ($c_n$)', 'Potential ($\phi$)']

for i, var in enumerate(variables):

    # Extract model predictions
    data_pinn = pinn_dict[var].reshape(400, 100)
    data_nn = nn_dict[var].reshape(400, 100)

    # Interpolate FDM truth onto the same grid for a 1:1 comparison
    data_fipy = griddata(fipy_pts, fipy_df[var].values, (X_grid, T_grid), method='linear')

    # Calculate Errors
    error_pinn = np.abs(data_fipy - data_pinn)
    error_nn = np.abs(data_fipy - data_nn)

    # Find global min/max for consistent color scales across predictions
    vmin_pred = np.nanmin([data_fipy, data_pinn, data_nn])
    vmax_pred = np.nanmax([data_fipy, data_pinn, data_nn])

    # Find global max for error scale (min is 0)
    vmax_err = np.nanmax([error_pinn, error_nn])

    # Column 0: FDM Truth
    im0 = axes[i, 0].contourf(X_grid, T_grid, data_fipy, levels=50, cmap='viridis', vmin=vmin_pred, vmax=vmax_pred)
    axes[i, 0].set_title(f'FDM Truth: {titles[i]}')
    fig.colorbar(im0, ax=axes[i, 0])

    # Column 1: PINN Prediction
    im1 = axes[i, 1].contourf(X_grid, T_grid, data_pinn, levels=50, cmap='viridis', vmin=vmin_pred, vmax=vmax_pred)
    axes[i, 1].set_title(f'PINN Pred: {titles[i]}')
    fig.colorbar(im1, ax=axes[i, 1])

    # Column 2: Standard NN Prediction
    im2 = axes[i, 2].contourf(X_grid, T_grid, data_nn, levels=50, cmap='viridis', vmin=vmin_pred, vmax=vmax_pred)
    axes[i, 2].set_title(f'NN Pred: {titles[i]}')
    fig.colorbar(im2, ax=axes[i, 2])

    # Column 3: PINN Absolute Error
    im3 = axes[i, 3].contourf(X_grid, T_grid, error_pinn, levels=50, cmap='magma', vmin=0, vmax=vmax_err)
    axes[i, 3].set_title(f'PINN Abs Error')
    fig.colorbar(im3, ax=axes[i, 3])

    # Column 4: Standard NN Absolute Error
    im4 = axes[i, 4].contourf(X_grid, T_grid, error_nn, levels=50, cmap='magma', vmin=0, vmax=vmax_err)
    axes[i, 4].set_title(f'NN Abs Error')
    fig.colorbar(im4, ax=axes[i, 4])

# --- 3. Formatting ---
for ax in axes[-1, :]: ax.set_xlabel('Position (x)')
for ax in axes[:, 0]:  ax.set_ylabel('Time (t)')

plt.tight_layout()
plt.show()

## 3. Results and Conclusion

The contour plots above compare the FDM ground truth, the Physics-Informed Neural Network (PINN), and the standard Neural Network (NN).

**Key Observations:**

1. **Superior Accuracy of Supervised Learning:** The standard Neural Network (NN) was much more accurate than the PINN. The highest error for the NN was only about 0.05, while the PINN reached errors as high as 0.27.
2. **Architecture Capability:** Because the standard NN performed so well, it proves that our network architecture is definitely capable of solving these equations. The model structure itself is working correctly.
3. **PINN Optimization Challenges:** The PINN captures the general "shape" of the ion movement, but it struggles with exact numbers. This shows that learning strictly from physics laws is a much harder task for the optimizer than learning directly from data.
4. **Potential Field Accuracy:** The standard NN was nearly perfect at reconstructing the electrical potential (phi). The PINN showed much higher error in the middle of the spatial domain, meaning it likely needs more training time or better settings to match the benchmark.

**Conclusion:**
In this experiment, the standard NN was the clear winner in terms of precision. This proves our chosen architecture is strong enough to handle the PNP system, but it also shows that the PINN requires more tuning to reach the same level of accuracy as a data-driven model.